In [1]:
# ======================================================================================
# SETUP: run this cell first. It works on macOS, Windows and Linux.
# It (1) checks the Python version, (2) installs the libraries below into the Python
# that runs this notebook, (3) imports everything the notebook uses, and (4) checks that
# the data files are next to the notebook.
#
# Libraries used (exact versions this notebook was tested with):
#   numpy 2.2.6, pandas 2.3.3, scikit-learn 1.7.2, scipy 1.15.3 (needed by scikit-learn),
#   jinja2 3.1.6 (needed by pandas to show the formatted tables)
# Everything else (json, os, subprocess, sys) is part of Python itself.
# ======================================================================================
import json
import os
import subprocess
import sys

INSTALL_LIBRARIES = True        # set to False if the libraries are already installed

if sys.version_info < (3, 10):
    raise RuntimeError(f"Python {sys.version.split()[0]} is too old: scikit-learn 1.7 needs Python 3.10 or newer. "
                       "On a Mac, install Python 3.12 from python.org (or with Homebrew) and select it as the kernel.")
if sys.version_info < (3, 14):
    REQUIREMENTS = ["numpy==2.2.6", "pandas==2.3.3", "scikit-learn==1.7.2", "scipy==1.15.3", "jinja2==3.1.6"]
else:                           # Python 3.14+: numpy 2.2 has no installer for it, so use newer compatible versions
    REQUIREMENTS = ["numpy>=2.3,<3", "pandas>=2.3.3,<3", "scikit-learn==1.7.2", "jinja2>=3.1"]
if INSTALL_LIBRARIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *REQUIREMENTS])

import numpy as np
import pandas as pd
import jinja2                                   # noqa: F401  (pandas uses it for .style tables)
import sklearn
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from IPython.display import display

DATA_FILES = ["game_info.csv.gz", "feature_groups.json", "train.csv.gz", "test.csv.gz", "validation.csv.gz",
              "video_game_sales_and_ratings.csv.gz"]
missing = [f for f in DATA_FILES if not os.path.exists(os.path.join("data", f))]
if missing:
    raise FileNotFoundError(f"Missing {missing} in {os.path.abspath('data')}. Open this notebook from the "
                            "final_research folder, so that the data folder sits next to it.")

RANDOM_STATE = 42
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print(f"Python {sys.version.split()[0]} on {sys.platform} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scikit-learn {sklearn.__version__}")
if INSTALL_LIBRARIES and (np.__version__, pd.__version__) != ("2.2.6", "2.3.3") and sys.version_info < (3, 14):
    print("A library version changed while this kernel was running: restart the kernel and run all cells again.")
print("Setup done: all libraries imported and all data files found.")

Python 3.10.11 on win32 | numpy 2.2.6 | pandas 2.3.3 | scikit-learn 1.7.2
Setup done: all libraries imported and all data files found.


# Final Research Implementation: Forecasting Steam Game Owners Before Release

**Hongyu Guo · DS 340W, Group 41**

**Research question.** How well can the number of owners of a Steam game be predicted using only information that exists on its release day? Which kind of information helps the most?

**Hypothesis.**

- **H1:** Release-day information predicts owners better than guessing the same number for every game.
- **H2:** A game's history, meaning how well its predecessor and its studio's earlier games sold, adds more than extra store-page details.

This notebook runs the whole project from start to finish, one step at a time:

| Step | What happens |
|---|---|
| 1 | Re-run parent paper 1 and show why its perfect score cannot be used before release |
| 2 | Load the Steam data and apply the release-day rule |
| 3 | Build the game-history variables (the idea from parent paper 3) |
| 4 | Load the training and test sets (the validation set stays closed) |
| 5 | Baseline: the parent paper's method on Steam |
| 6 | Change one part of the method at a time (parent papers 2 and 3, plus my own changes) |
| 7 | Compare all models, and see which games the final model works for |
| 8 | Score the unseen validation set once |
| 9 | Check whether the final model is accurate enough to use |
| 10 | Conclusions |

**Parent papers.**

| # | Paper | What it does | What I used from it |
|---|---|---|---|
| 1 | Gray, Shalan and Kadlec, IEEE CICN 2024 | Linear Regression and a Decision Tree predict global sales; R² = 1.000 | The two models and the metrics; the baseline |
| 2 | Bondarenko, Cabrera and Fitzek, IEEE SmartNets 2026 | Log-linear models of how many Steam wishlists turn into sales | Log target, separate paid and free-to-play models, price bands, effects read as percentages |
| 3 | Schaer, Kourentzes and Fildes, EJOR 2026 | Forecasts a new game from the sales of the previous game in its franchise | Predecessor sales as an input, widened from franchises to developers and publishers |

## Step 1. Why the parent paper's result cannot be used before release

Parent paper 1 trained Linear Regression and a Decision Tree on the Kaggle *Video Game Sales and Ratings* data and reported R² = 1.000. The cell below repeats its setup (10-fold cross-validation, mean imputation, Orange's tree settings) three times:

1. with the paper's own inputs;
2. without the four regional sales columns;
3. also without the row index.

The full reproduction of the paper's tables is in [`../parent_paper_1_implementation/`](../parent_paper_1_implementation/).

In [2]:
paper = pd.read_csv("data/video_game_sales_and_ratings.csv.gz").reset_index()   # the paper used the row index as an input
REGIONAL = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]
PAPER_INPUTS = ["index", "Year_of_Release"] + REGIONAL + ["Critic_Score", "Critic_Count", "User_Count"]
input_sets = {
    "1. The paper's inputs": PAPER_INPUTS,
    "2. Without regional sales": [c for c in PAPER_INPUTS if c not in REGIONAL],
    "3. Without regional sales and row index": [c for c in PAPER_INPUTS if c not in REGIONAL + ["index"]],
}
paper_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(min_samples_leaf=2, min_samples_split=5, max_depth=100, random_state=5078),
}
cv = KFold(n_splits=10, shuffle=True, random_state=5078)
rows = {}
for label, inputs in input_sets.items():
    rows[label] = {name: r2_score(paper["Global_Sales"],
                                  cross_val_predict(make_pipeline(SimpleImputer(), model), paper[inputs], paper["Global_Sales"], cv=cv))
                   for name, model in paper_models.items()}
print(f"Parent paper data: {len(paper):,} games. Rows are sorted by global sales: first row = {paper.loc[0, 'Name']}")
lr = LinearRegression().fit(SimpleImputer().fit_transform(paper[PAPER_INPUTS]), paper["Global_Sales"])
print("Linear Regression weights on the regional sales:", {c: round(float(w), 3) for c, w in zip(REGIONAL, lr.coef_[2:6])})
pd.DataFrame(rows).T.style.format("{:.3f}").set_caption("R² for global sales, 10-fold cross-validation (parent paper's data)")

Parent paper data: 16,719 games. Rows are sorted by global sales: first row = Wii Sports
Linear Regression weights on the regional sales: {'NA_Sales': 1.0, 'EU_Sales': 1.0, 'JP_Sales': 1.0, 'Other_Sales': 1.0}


,Linear Regression,Decision Tree
1. The paper's inputs,1.000,0.912
2. Without regional sales,0.217,0.923
3. Without regional sales and row index,0.098,-0.104


**Reading the result.**

- **The perfect score is an accounting identity.** Global sales are the sum of the four regional sales columns, and Linear Regression gives each of them a weight of almost exactly 1.
- **The row index leaks too.** The file is sorted from best-selling to worst-selling, so the row number is a sales rank. Without regional sales, the Decision Tree still scores about 0.92 by reading the index.
- **Without both leaks, the method explains almost nothing.** Also, critic and user counts only exist after release.

So my project keeps the paper's models and metrics but allows **only information that exists on release day**.

## Step 2. Steam data and the release-day rule

**Source.** The [Steam Games Dataset](https://www.kaggle.com/datasets/fronkongames/steam-games-dataset) on Kaggle (version 49, MIT license). It joins Steam store pages with SteamSpy owner estimates. Kaggle no longer offers version 49 for download, so the cleaned data is saved in `data/`. The cleaning code is in [`new_analysis/notebooks/01`](../new_analysis/notebooks/01_preprocessing_and_transformation.ipynb).

**Cleaning.**

| Step | Games left |
|---|---|
| All applications in the raw file | 141,900 |
| Remove playtests and demos | 133,688 |
| Remove software that is not a game | 132,243 |
| Remove games with no owner estimate | 101,630 |
| Keep games released by 30 June 2024 (at least one year of sales) | **76,727** |

**Target.** SteamSpy gives owners as a range, such as 20,000–50,000. Each game gets the midpoint of its range, and the models predict log(1 + owners).

**The release-day rule.** Anything recorded after launch is removed: reviews, user scores, playtime, player counts, user tags, DLC counts, the current (discounted) price, the "Early Access" genre and Steam features that Valve adds after release. What remains is **42 store-page variables**:

- price, release year, required age, number of interface languages and of fully voiced languages;
- Mac and Linux support, number of screenshots, description length;
- 15 genre tags: Indie, Casual, Action, Adventure, Simulation, Strategy, RPG, Free To Play, Sports, Racing, Massively Multiplayer, Violent, Gore, Education, Nudity;
- 18 Steam feature tags: Single-player, Steam Cloud, Full controller support, Multi-player, Partial Controller Support, PvP, Co-op, Steam Leaderboards, Online PvP, Shared/Split Screen, Tracked Controller Support, VR Only, Online Co-op, Shared/Split Screen PvP, Stats, Shared/Split Screen Co-op, Includes level editor, Captions available.

Later steps add 7 studio variables and 10 history variables, for 59 in total.

In [3]:
# keep_default_na=False: publishers literally named "NA" or "None" must stay names, not become missing values
games = pd.read_csv("data/game_info.csv.gz", parse_dates=["release_date"], keep_default_na=False,
                    na_values={"developer": [""], "publisher": [""]})
games["owners"] = (games["owners_millions"] * 1e6).round()
games["log_owners"] = np.log1p(games["owners"])
with open("data/feature_groups.json") as f:
    groups = json.load(f)

print(f"{len(games):,} games released {games['release_date'].dt.year.min()}-{games['release_date'].dt.year.max()}")
print(f"{games['developer'].nunique():,} developers, {games['publisher'].nunique():,} publishers")

ranges_table = pd.DataFrame({"Midpoint used": np.sort(games["owners"].unique())})
counts = games["owners"].value_counts()
ranges_table["Games"] = ranges_table["Midpoint used"].map(counts)
ranges_table["Share"] = ranges_table["Games"] / len(games)
ranges_table.style.format({"Midpoint used": "{:,.0f}", "Games": "{:,}", "Share": "{:.2%}"}).hide(axis="index").set_caption(
    "Owners per game: SteamSpy range midpoints")

76,727 games released 1997-2024
45,790 developers, 40,334 publishers


Midpoint used,Games,Share
"10,000","53,446",69.66%
"35,000","10,222",13.32%
"75,000","4,865",6.34%
"150,000","3,172",4.13%
"350,000","2,659",3.47%
"750,000","1,076",1.40%
"1,500,000",693,0.90%
"3,500,000",388,0.51%
"7,500,000",120,0.16%
"15,000,000",46,0.06%


**The target is extremely uneven.** About 70% of games sit in the lowest range (under 20,000 owners), and only about 1% reach a million. Any model can look accurate by predicting "small" for everything, which Step 9 checks.

## Step 3. Game-history variables (the idea from parent paper 3)

Parent paper 3 forecasts a new game from the lifetime sales of the previous game in the same franchise. It works only for franchises. I widened the idea to every Steam game by looking for the closest earlier game in this order:

1. **Series predecessor:** an earlier game with the same title stem (for example "Sniper Elite 4" → "sniperelite") from the same developer or publisher;
2. **Developer's previous game;**
3. **Publisher's previous game.**

**Ten variables** are built:

- log owners of each of the three predecessors, and years since each one came out (6 variables);
- mean and best log owners of all earlier games by the developer and by the publisher (4 variables).

Only games released *before* a game count as its history. A game with no earlier game keeps a missing value, which Gradient Boosting can handle.

In [4]:
NUMERAL = r"(?:[2-9]|1[0-9]|II|III|IV|V|VI|VII|VIII|IX|X)"
stem = games["name"].fillna("").str.split(r":|\s-\s", regex=True).str[0]                       # drop a subtitle
stem = stem.str.replace(rf"\s+(?:[0-9]+|{NUMERAL})\s*$", "", regex=True)                       # drop a trailing number
stem = stem.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
stem = stem.where(stem.str.len() >= 4)                                                         # very short stems are ambiguous
for key, owner in [("series_dev", "developer"), ("series_pub", "publisher")]:
    games[key] = np.where(stem.notna() & games[owner].notna(), stem.astype(str) + "|" + games[owner].astype(str), None)


def previous_game(key):
    """For every game: the most recent game with the same key released strictly earlier."""
    have = games.dropna(subset=[key])[["appid", key, "release_date", "log_owners"]].sort_values("release_date")
    earlier = have.rename(columns={"appid": "prev_appid", "log_owners": "prev_log_owners"}).assign(prev_date=have["release_date"])
    m = pd.merge_asof(have[["appid", key, "release_date"]], earlier[[key, "release_date", "prev_appid", "prev_log_owners", "prev_date"]],
                      on="release_date", by=key, allow_exact_matches=False)
    m["gap_years"] = (m["release_date"] - m["prev_date"]).dt.days / 365.25
    return games[["appid"]].merge(m[["appid", "prev_appid", "prev_log_owners", "gap_years"]], on="appid", how="left")


def earlier_sales(key):
    """Mean and best log owners of all games with the same key released strictly earlier."""
    have = games.dropna(subset=[key])
    per_day = have.groupby([key, "release_date"])["log_owners"].agg(n="size", total="sum", top="max").reset_index()
    per_day = per_day.sort_values("release_date")
    by_key = per_day.groupby(key)
    per_day["cum_n"], per_day["cum_total"], per_day["cum_top"] = by_key["n"].cumsum(), by_key["total"].cumsum(), by_key["top"].cummax()
    left = games[[key, "release_date"]].reset_index().dropna(subset=[key]).sort_values("release_date")
    m = pd.merge_asof(left, per_day[[key, "release_date", "cum_n", "cum_total", "cum_top"]], on="release_date", by=key,
                      allow_exact_matches=False).set_index("index").reindex(games.index)
    return m["cum_total"] / m["cum_n"], m["cum_top"]


by_dev, by_pub = previous_game("series_dev"), previous_game("series_pub")
use_pub = by_dev["prev_appid"].isna() | (by_pub["gap_years"] < by_dev["gap_years"])     # the more recent series match
series = by_dev.where(~use_pub, by_pub)
developer, publisher = previous_game("developer"), previous_game("publisher")

history = pd.DataFrame({"appid": games["appid"]})
for name, frame in [("series", series), ("dev", developer), ("pub", publisher)]:
    history[f"{name}_prev_log_owners"], history[f"{name}_gap_years"] = frame["prev_log_owners"], frame["gap_years"]
history["dev_prior_mean_log_owners"], history["dev_prior_max_log_owners"] = earlier_sales("developer")
history["pub_prior_mean_log_owners"], history["pub_prior_max_log_owners"] = earlier_sales("publisher")
history["analogy_type"] = np.select(
    [series["prev_appid"].notna(), developer["prev_appid"].notna(), publisher["prev_appid"].notna()],
    ["Series predecessor", "Developer's previous game", "Publisher's previous game"], "No predecessor")
closest = series.where(series["prev_appid"].notna(), developer.where(developer["prev_appid"].notna(), publisher))
history["closest_prev_log_owners"] = closest["prev_log_owners"]
HISTORY = groups["PREDECESSOR"] + groups["STUDIO_AVERAGES"]
print("History variables built:", HISTORY)

History variables built: ['series_prev_log_owners', 'series_gap_years', 'dev_prev_log_owners', 'dev_gap_years', 'pub_prev_log_owners', 'pub_gap_years', 'dev_prior_mean_log_owners', 'dev_prior_max_log_owners', 'pub_prior_mean_log_owners', 'pub_prior_max_log_owners']


In [5]:
KINDS = ["Series predecessor", "Developer's previous game", "Publisher's previous game", "No predecessor"]
h = history.assign(log_owners=games["log_owners"], big=games["owners"] >= 100_000)
coverage = pd.DataFrame({
    "Games": h.groupby("analogy_type").size(),
    "Share of games": h.groupby("analogy_type").size() / len(h),
    "Correlation with the predecessor (log owners)": h.groupby("analogy_type").apply(
        lambda g: g["log_owners"].corr(g["closest_prev_log_owners"]), include_groups=False),
    "Share with 100,000+ owners": h.groupby("analogy_type")["big"].mean(),
}).reindex(KINDS)
display(coverage.style.format({"Games": "{:,}", "Share of games": "{:.1%}", "Correlation with the predecessor (log owners)": "{:.2f}",
                               "Share with 100,000+ owners": "{:.1%}"}, na_rep="-").set_caption("Which history each game has"))

# Examples: the most-owned recent games that have a series predecessor
names = games.set_index("appid")["name"]
example = games[["appid", "name", "release_date", "owners"]].assign(prev_appid=series["prev_appid"])
example = example[example["prev_appid"].notna() & (example["release_date"].dt.year >= 2019)].nlargest(6, "owners")
example["Predecessor"] = example["prev_appid"].map(names)
example["Predecessor's owners"] = example["prev_appid"].map(games.set_index("appid")["owners"])
example[["name", "release_date", "Predecessor", "Predecessor's owners", "owners"]].rename(
    columns={"name": "Game", "release_date": "Released", "owners": "Owners"}).style.format(
    {"Released": "{:%Y-%m-%d}", "Predecessor's owners": "{:,.0f}", "Owners": "{:,.0f}"}).hide(axis="index").set_caption(
    "Examples of series predecessors found by the title-stem rule")

,Games,Share of games,Correlation with the predecessor (log owners),"Share with 100,000+ owners"
analogy_type,,,,
Series predecessor,"5,284",6.9%,0.78,17.5%
Developer's previous game,"25,349",33.0%,0.56,11.4%
Publisher's previous game,"6,377",8.3%,0.51,22.1%
No predecessor,"39,717",51.8%,-,7.5%


Game,Released,Predecessor,Predecessor's owners,Owners
Call of Duty®,2022-10-27,Call of Duty®: Modern Warfare® II,"10,000","75,000,000"
Battlefield™ 2042,2021-11-19,Battlefield™ V,"15,000,000","15,000,000"
Mount & Blade II: Bannerlord,2022-10-25,Mount & Blade: With Fire & Sword,"1,500,000","15,000,000"
The Sims™ 4,2020-06-18,The Sims™ 3,"3,500,000","15,000,000"
Battlefield™ V,2020-10-22,Battlefield™ 1,"7,500,000","15,000,000"
Halo: The Master Chief Collection,2019-12-03,Halo: Spartan Strike,"350,000","7,500,000"


**Reading the result.**

- **About half of all games (52%) have no earlier game at all.** They are a studio's first release in this data, so the history variables cannot help them.
- **A series predecessor is the closest guide.** Its owners correlate about 0.78 with the new game's owners on the log scale, against about 0.56 for the developer's previous game.
- Games with any earlier game reach 100,000 owners more often (11–22%) than games with no history (7.5%).
- **The title rule is simple and sometimes wrong.** Most matches are true sequels, such as The Sims 4 ← The Sims 3. But "Call of Duty®" (2022) is matched to another store page of the same franchise with few owners, and the Halo collection to a small spin-off. These mistakes add noise to the history variables.

## Step 4. Training, test and validation sets (70% / 20% / 10%)

The 76,727 games were split once, in Week 11, at random with seed 42 and stratified by owner range, into three files (see [`../novel_approach/`](../novel_approach/), Section 2):

| File | Share | Games | Used for |
|---|---|---|---|
| `data/train.csv.gz` | 70% | 53,708 | Fitting every model |
| `data/test.csv.gz` | 20% | 15,346 | Comparing models and choosing which changes to keep |
| `data/validation.csv.gz` | 10% | 7,673 | Unseen: scored once, in Step 8, after the final model is chosen |

Following the course, *test* is the set checked during development and *validation* is the unseen final set. The validation file is not opened until Step 8.

Each file already holds all 59 variables. The cell checks that the history variables in the files are exactly the ones built in Step 3.

In [6]:
BASE = groups["BASE"]                                     # 42 store-page variables
STORE_PAGE = BASE + groups["STUDIO_AND_LEAD_TIME"]        # + 7 studio variables = 49
ANALOGY = STORE_PAGE + HISTORY                            # + 10 history variables = 59
EDGES = [0, 10, 20, 30, 40, 60, 80, np.inf]               # price bands (parent paper 2), $0-9.99 is the reference
PRICE_BANDS = [f"price_{lo}_{hi}" if np.isfinite(hi) else f"price_{lo}_plus" for lo, hi in zip(EDGES[1:-1], EDGES[2:])]


def add_columns(frame):
    """Target on the log scale and the price-band indicators."""
    frame = frame.copy()
    frame["owners"] = (frame["owners_millions"] * 1e6).round()
    frame["log_owners"] = np.log1p(frame["owners"])
    for name, lo, hi in zip(PRICE_BANDS, EDGES[1:-1], EDGES[2:]):
        frame[name] = ((frame["price"] >= lo) & (frame["price"] < hi)).astype(int)
    return frame


train = add_columns(pd.read_csv("data/train.csv.gz"))
test = add_columns(pd.read_csv("data/test.csv.gz"))
assert not set(train["appid"]) & set(test["appid"]), "a game is in both sets"

for name, frame in [("training", train), ("test", test)]:
    rebuilt = frame[["appid"]].merge(history, on="appid", how="left")
    same = all(np.allclose(frame[c].fillna(-1), rebuilt[c].fillna(-1)) for c in HISTORY) and (frame["analogy_type"] == rebuilt["analogy_type"]).all()
    print(f"{name} set: {len(frame):,} games | history variables identical to Step 3: {same}")

print(f"Variables: store page {len(BASE)}, + studio {len(STORE_PAGE)}, + history {len(ANALOGY)}")

training set: 53,708 games | history variables identical to Step 3: True
test set: 15,346 games | history variables identical to Step 3: True
Variables: store page 42, + studio 49, + history 59


## Step 5. Baseline: the parent paper's method on Steam

Every model is fitted on the training set and scored in three ways:

1. **Random folds:** 5-fold cross-validation inside the training set.
2. **Test set:** the 20% test set.
3. **One year ahead:** each of 2022, 2023 and 2024 is predicted by a model fitted only on earlier training-set games, which is how a real forecast would be made. This is the main evidence.

**Main measure: R² on log owners.** On raw owners, R² is ruled by a handful of blockbusters. A score below 0 means worse than predicting the average for every game.

The baseline is parent paper 1's Linear Regression and Decision Tree on raw owners (in millions), with the 42 store-page variables.

In [7]:
folds = list(KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(train))
AHEAD_YEARS = (2022, 2023, 2024)


class PaidFreeModel:
    """Parent paper 2: one model for paid games and another for free-to-play games."""

    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        paid = (X["price"] > 0).to_numpy()
        self.models = {seg: clone(self.model).fit(X[paid == seg], y[paid == seg]) for seg in (True, False)}
        return self

    def predict(self, X):
        paid = (X["price"] > 0).to_numpy()
        pred = np.zeros(len(X))
        for seg in (True, False):
            if (paid == seg).any():
                pred[paid == seg] = self.models[seg].predict(X[paid == seg])
        return pred


def prepare(frame, features, linear):
    """Linear models cannot use missing values: fill them with 0 and add a 'missing' indicator."""
    X = frame[features]
    if not linear:
        return X
    missing = [c for c in features if train[c].isna().any()]
    return pd.concat([X.fillna(0), X[missing].isna().astype(int).add_suffix("_missing")], axis=1)


def fit_predict(spec, fit_frame, score_frame):
    """Fit on fit_frame and return predicted log owners for score_frame."""
    model = PaidFreeModel(spec["model"]) if spec["paid_free"] else clone(spec["model"])
    linear = isinstance(spec["model"], LinearRegression)
    X_fit, X_score = prepare(fit_frame, spec["features"], linear), prepare(score_frame, spec["features"], linear)
    if spec["target"] == "raw":
        pred = model.fit(X_fit, fit_frame["owners"] / 1e6).predict(X_score) * 1e6
        return np.log1p(np.clip(pred, 0, None))
    return model.fit(X_fit, fit_frame["log_owners"]).predict(X_score)


results, specs = [], {}


def evaluate(name, block, source, model, features, target="log", paid_free=False):
    spec = {"model": model, "features": features, "target": target, "paid_free": paid_free}
    oof = np.zeros(len(train))
    for tr, va in folds:
        oof[va] = fit_predict(spec, train.iloc[tr], train.iloc[va])
    ahead = [r2_score(train.loc[train["release_year"] == y, "log_owners"],
                      fit_predict(spec, train[train["release_year"] < y], train[train["release_year"] == y]))
             for y in AHEAD_YEARS]
    test_log = fit_predict(spec, train, test)
    row = {"Model": name, "Part changed": block, "Source": source, "Variables": len(features),
           "Random folds: R² (log)": r2_score(train["log_owners"], oof),
           "Test: R² (log)": r2_score(test["log_owners"], test_log),
           "One year ahead: R² (log)": np.mean(ahead),
           "Test: MAE (owners)": mean_absolute_error(test["owners"], np.expm1(test_log))}
    results.append(row)
    specs[name] = spec
    print(f"{name}\n  random folds {row['Random folds: R² (log)']:.3f} | test {row['Test: R² (log)']:.3f} | "
          f"one year ahead {row['One year ahead: R² (log)']:.3f} | test MAE {row['Test: MAE (owners)']:,.0f} owners")


evaluate("B1: Linear Regression, raw owners", "-", "Parent paper 1", LinearRegression(), BASE, target="raw")
evaluate("B2: Decision Tree, raw owners", "-", "Parent paper 1", DecisionTreeRegressor(random_state=RANDOM_STATE), BASE, target="raw")

B1: Linear Regression, raw owners
  random folds -25.473 | test -25.577 | one year ahead -70.648 | test MAE 198,171 owners


B2: Decision Tree, raw owners
  random folds -0.193 | test -0.215 | one year ahead -0.875 | test MAE 193,989 owners


**Without the leaking inputs, the parent paper's method fails on Steam.** Both models score below zero. Linear Regression predicts negative owners for many small games, and the Decision Tree chases the few blockbusters.

## Step 6. Changing one part of the method at a time

Each change replaces one part of the pipeline (the target, the segments, the inputs or the model) and keeps the rest.

**From parent paper 2 (M1–M3, M5):**

- **M1:** predict log owners instead of raw owners (a log-linear model);
- **M2:** separate models for paid and free-to-play games;
- **M3:** price bands ($10–19.99 up to $80+) instead of a straight-line price effect.

In [8]:
evaluate("M1: Linear Regression, log owners", "Target", "Parent paper 2", LinearRegression(), BASE)
evaluate("M2: + separate paid and free-to-play models", "Segments", "Parent paper 2", LinearRegression(), BASE, paid_free=True)
evaluate("M3: + price bands", "Inputs", "Parent paper 2", LinearRegression(), BASE + PRICE_BANDS, paid_free=True)

M1: Linear Regression, log owners
  random folds 0.288 | test 0.290 | one year ahead 0.055 | test MAE 104,258 owners


M2: + separate paid and free-to-play models
  random folds 0.294 | test 0.296 | one year ahead 0.063 | test MAE 103,926 owners


M3: + price bands
  random folds 0.313 | test 0.311 | one year ahead 0.089 | test MAE 103,394 owners


**Effects as percentages (parent paper 2's way of reporting).** For a coefficient *b* on log owners, one more unit goes with a 100 × (e^b − 1)% change in owners. These are associations, not causes: they are not advice to raise prices.

In [9]:
SHOWN = {"price": "Price (per $10)", "n_languages": "Each supported language", "mac": "Mac support",
         "cat_Multi_player": "Multi-player", "cat_Online_Co_op": "Online co-op", "genre_Indie": "Indie genre tag"}
effects = {}
for segment, frame in [("Paid games", train[train["price"] > 0]), ("Free-to-play games", train[train["price"] == 0])]:
    coef = pd.Series(LinearRegression().fit(frame[BASE], frame["log_owners"]).coef_, index=BASE)
    coef["price"] *= 10
    effects[segment] = 100 * (np.exp(coef[list(SHOWN)]) - 1)
effects = pd.DataFrame(effects).rename(index=SHOWN)
effects.loc["Price (per $10)", "Free-to-play games"] = np.nan
effects.style.format("{:+.1f}%", na_rep="-").set_caption("Change in expected owners, other store-page variables held fixed")

,Paid games,Free-to-play games
Price (per $10),+7.9%,-
Each supported language,+1.6%,+6.5%
Mac support,+11.2%,+21.5%
Multi-player,+37.8%,+42.4%
Online co-op,+44.4%,+8.3%
Indie genre tag,-14.2%,-20.4%


**Model change (my addition):**

- **M4:** Gradient Boosting (scikit-learn's `HistGradientBoostingRegressor`, default settings) on log owners. It builds many small trees, each fixing the errors of the ones before, so it can capture curves and interactions.
- **M5:** M4 with parent paper 2's paid and free-to-play split.

In [10]:
evaluate("M4: Gradient Boosting, log owners", "Model", "My addition", HistGradientBoostingRegressor(random_state=RANDOM_STATE), BASE)
evaluate("M5: M4 + separate paid and free-to-play models", "Segments", "Parent paper 2",
         HistGradientBoostingRegressor(random_state=RANDOM_STATE), BASE, paid_free=True)

M4: Gradient Boosting, log owners
  random folds 0.435 | test 0.431 | one year ahead 0.244 | test MAE 94,933 owners


M5: M4 + separate paid and free-to-play models
  random folds 0.432 | test 0.431 | one year ahead 0.238 | test MAE 97,619 owners


The split does not help Gradient Boosting, whose trees already treat paid and free-to-play games differently, so M5 is **not kept**.

**New inputs:**

- **M6 (my addition):** 7 studio variables: earlier releases by the developer and by the publisher, first game or not, self-published or not, number of developers, and how many days the store page was up before release.
- **M7 (parent paper 3):** the 10 history variables from Step 3.
- **Check:** the history variables added to the best linear model (M3), to see whether the gain comes from the information or from Gradient Boosting.

In [11]:
evaluate("M6: M4 + studio variables", "Inputs", "My addition", HistGradientBoostingRegressor(random_state=RANDOM_STATE), STORE_PAGE)
evaluate("M7: + history variables (final model)", "Inputs", "Parent paper 3",
         HistGradientBoostingRegressor(random_state=RANDOM_STATE), ANALOGY)
evaluate("Check: M3 + history variables", "Inputs", "Parent paper 3", LinearRegression(), ANALOGY + PRICE_BANDS, paid_free=True)

M6: M4 + studio variables
  random folds 0.457 | test 0.455 | one year ahead 0.257 | test MAE 93,818 owners


M7: + history variables (final model)
  random folds 0.499 | test 0.495 | one year ahead 0.312 | test MAE 92,107 owners


Check: M3 + history variables
  random folds 0.411 | test 0.400 | one year ahead 0.228 | test MAE 98,578 owners


## Step 7. Comparing all models

In [12]:
table = pd.DataFrame(results).set_index("Model")
table.style.format({"Variables": "{:d}", "Random folds: R² (log)": "{:.3f}", "Test: R² (log)": "{:.3f}",
                    "One year ahead: R² (log)": "{:.3f}", "Test: MAE (owners)": "{:,.0f}"}).set_caption(
    "Every model on the same evaluations")

,Part changed,Source,Variables,Random folds: R² (log),Test: R² (log),One year ahead: R² (log),Test: MAE (owners)
Model,,,,,,,
"B1: Linear Regression, raw owners",-,Parent paper 1,42,-25.473,-25.577,-70.648,"198,171"
"B2: Decision Tree, raw owners",-,Parent paper 1,42,-0.193,-0.215,-0.875,"193,989"
"M1: Linear Regression, log owners",Target,Parent paper 2,42,0.288,0.290,0.055,"104,258"
M2: + separate paid and free-to-play models,Segments,Parent paper 2,42,0.294,0.296,0.063,"103,926"
M3: + price bands,Inputs,Parent paper 2,48,0.313,0.311,0.089,"103,394"
"M4: Gradient Boosting, log owners",Model,My addition,42,0.435,0.431,0.244,"94,933"
M5: M4 + separate paid and free-to-play models,Segments,Parent paper 2,42,0.432,0.431,0.238,"97,619"
M6: M4 + studio variables,Inputs,My addition,49,0.457,0.455,0.257,"93,818"
M7: + history variables (final model),Inputs,Parent paper 3,59,0.499,0.495,0.312,"92,107"


**Which games does the final model work for?** The one-year-ahead predictions of the final model (M7) for 2022–2024 are split by the history a game has, and by paid or free-to-play. M6 (no history variables) is shown alongside to see what the history adds.

In [13]:
FINAL = "M7: + history variables (final model)"
parts = []
for year in AHEAD_YEARS:
    earlier, scored = train[train["release_year"] < year], train[train["release_year"] == year].copy()
    scored["pred"] = fit_predict(specs[FINAL], earlier, scored)
    scored["pred_m6"] = fit_predict(specs["M6: M4 + studio variables"], earlier, scored)
    parts.append(scored)
ahead = pd.concat(parts)


def mean_yearly_r2(frame, col):
    return np.mean([r2_score(g["log_owners"], g[col]) for _, g in frame.groupby("release_year")])


subsets = {"All games": ahead["price"] >= 0, **{k: ahead["analogy_type"] == k for k in KINDS},
           "Paid games": ahead["price"] > 0, "Free-to-play games": ahead["price"] == 0}
by_kind = pd.DataFrame([{"Games": name, "Games scored": m.sum(), "Share": m.mean(),
                         "M6 (no history): R² (log)": mean_yearly_r2(ahead[m], "pred_m6"),
                         "M7 (final): R² (log)": mean_yearly_r2(ahead[m], "pred")} for name, m in subsets.items()]).set_index("Games")
by_kind.style.format({"Games scored": "{:,}", "Share": "{:.1%}", "M6 (no history): R² (log)": "{:.3f}",
                      "M7 (final): R² (log)": "{:.3f}"}).set_caption("One year ahead (2022-2024), by kind of game")

,Games scored,Share,M6 (no history): R² (log),M7 (final): R² (log)
Games,,,,
All games,"20,043",100.0%,0.257,0.312
Series predecessor,"1,311",6.5%,0.465,0.589
Developer's previous game,"6,885",34.4%,0.299,0.383
Publisher's previous game,"1,368",6.8%,0.289,0.375
No predecessor,"10,479",52.3%,0.095,0.096
Paid games,"18,522",92.4%,0.281,0.342
Free-to-play games,"1,521",7.6%,-0.041,-0.050


**Reading the results.**

- **Parent paper 2's changes help the linear model** (random folds 0.288 → 0.313), and **Gradient Boosting** gives the largest single jump (0.435).
- **The history variables from parent paper 3 add the most new information.** One year ahead, R² rises from 0.257 (M6) to 0.312 (M7). They also lift Linear Regression from 0.313 to 0.411, so the gain comes from the information, not from the model.
- **But the gain is concentrated in games that have history.** With a series predecessor, the final model reaches about 0.59 one year ahead. With no earlier game, about half of all games, it stays near 0.10, almost the same as without history variables. Free-to-play games cannot be forecast at all.

**Which history inputs matter, year by year?** The one-year-ahead check is repeated for each year, adding to M6 either the 6 predecessor variables, the 4 studio-sales variables, or both (M7). The last table compares the final model with parent paper 3's simple benchmark: copy the series predecessor's owners.

In [14]:
history_sets = {"M6 (no history)": STORE_PAGE, "+ predecessor (6)": STORE_PAGE + groups["PREDECESSOR"],
                "+ studio sales (4)": STORE_PAGE + groups["STUDIO_AVERAGES"], "M7 (final)": ANALOGY}
by_year = {}
for year in AHEAD_YEARS:
    earlier, scored = train[train["release_year"] < year], train[train["release_year"] == year]
    row = {"Games": len(scored), "Under 20,000 owners": (scored["owners"] < 20_000).mean()}
    for name, feats in history_sets.items():
        pred = HistGradientBoostingRegressor(random_state=RANDOM_STATE).fit(earlier[feats], earlier["log_owners"]).predict(scored[feats])
        row[name] = r2_score(scored["log_owners"], pred)
    by_year[year] = row
by_year = pd.DataFrame(by_year).T
by_year.loc["Mean"] = by_year.mean()
by_year.loc["Mean", ["Games", "Under 20,000 owners"]] = np.nan
display(by_year.style.format({"Games": "{:,.0f}", "Under 20,000 owners": "{:.1%}", **{k: "{:.3f}" for k in history_sets}},
                             na_rep="-").set_caption("One year ahead: R² by year and history inputs"))

series_games = ahead[ahead["analogy_type"] == "Series predecessor"]
pd.DataFrame({"R² (log owners), 2022-2024 pooled": {
    "Copy the predecessor's owners (paper 3's benchmark)": r2_score(series_games["log_owners"], series_games["series_prev_log_owners"]),
    "Final model (M7)": r2_score(series_games["log_owners"], series_games["pred"])}}).style.format("{:.3f}").set_caption(
    f"Games with a series predecessor ({len(series_games):,} games)")

,Games,"Under 20,000 owners",M6 (no history),+ predecessor (6),+ studio sales (4),M7 (final)
2022,"7,039",75.5%,0.271,0.297,0.318,0.322
2023,"8,070",80.7%,0.215,0.266,0.281,0.284
2024,"4,934",84.6%,0.284,0.323,0.335,0.332
Mean,-,-,0.257,0.295,0.311,0.312


,"R² (log owners), 2022-2024 pooled"
Copy the predecessor's owners (paper 3's benchmark),0.225
Final model (M7),0.587


The four studio-sales variables give almost the whole gain on their own (0.311 against 0.312): a studio's average over all its earlier games is a steadier guide than its single last game. For sequels, the final model (0.587) is far better than simply copying the predecessor's owners (0.225).

## Step 8. Validation (unseen data, scored once)

Development is finished and M7 is the final model. Only now is the validation file opened. Each model is fitted on the training set as before and scored once on the 7,673 validation games. Nothing is changed after seeing these numbers.

In [15]:
validation = add_columns(pd.read_csv("data/validation.csv.gz"))
assert not set(validation["appid"]) & (set(train["appid"]) | set(test["appid"]))
rebuilt = validation[["appid"]].merge(history, on="appid", how="left")
assert all(np.allclose(validation[c].fillna(-1), rebuilt[c].fillna(-1)) for c in HISTORY)
print(f"Validation set: {len(validation):,} games, none of them in the training or test set")

rows = []
for name in ["B1: Linear Regression, raw owners", "B2: Decision Tree, raw owners", "M3: + price bands",
             "M4: Gradient Boosting, log owners", FINAL]:
    val_log = fit_predict(specs[name], train, validation)
    rows.append({"Model": name, "Test: R² (log)": table.loc[name, "Test: R² (log)"],
                 "Validation: R² (log)": r2_score(validation["log_owners"], val_log),
                 "Validation: MAE (owners)": mean_absolute_error(validation["owners"], np.expm1(val_log))})
    if name == FINAL:
        validation["pred"] = val_log
display(pd.DataFrame(rows).set_index("Model").style.format(
    {"Test: R² (log)": "{:.3f}", "Validation: R² (log)": "{:.3f}", "Validation: MAE (owners)": "{:,.0f}"}).set_caption(
    "Unseen validation set"))

val_kind = pd.DataFrame([{"Games": k, "Games scored": (validation["analogy_type"] == k).sum(),
                          "Validation: R² (log)": r2_score(validation.loc[validation["analogy_type"] == k, "log_owners"],
                                                           validation.loc[validation["analogy_type"] == k, "pred"])} for k in KINDS])
val_kind.set_index("Games").style.format({"Games scored": "{:,}", "Validation: R² (log)": "{:.3f}"}).set_caption(
    "Final model on the validation set, by kind of game")

Validation set: 7,673 games, none of them in the training or test set


,Test: R² (log),Validation: R² (log),Validation: MAE (owners)
Model,,,
"B1: Linear Regression, raw owners",-25.577,-25.921,"195,404"
"B2: Decision Tree, raw owners",-0.215,-0.219,"196,448"
M3: + price bands,0.311,0.314,"102,562"
"M4: Gradient Boosting, log owners",0.431,0.440,"95,099"
M7: + history variables (final model),0.495,0.506,"89,381"


,Games scored,Validation: R² (log)
Games,,
Series predecessor,559,0.771
Developer's previous game,"2,506",0.552
Publisher's previous game,625,0.468
No predecessor,"3,983",0.363


**The results hold on unseen games.** The final model scores about 0.51 on the validation set against 0.495 on the test set, every model keeps its rank, and the baseline stays below zero. The same pattern by kind of game appears: series predecessors are forecast best, games with no history worst. (These are games from the same years as the training set, so the scores are higher than one year ahead.)

## Step 9. Is the final model accurate enough to use?

R² ≈ 0.5 sounds reasonable, but R² only says how well the model *ranks* games compared with guessing the average. A developer wants to know: *if the model says my game will get X owners, how close will it be?* Three checks answer this, on the one-year-ahead forecasts (the realistic case) and on the validation set.

**Check 1: compare with a constant guess.** The simplest possible forecast predicts the training median, 10,000 owners, for every game. Two measures:

- **Within 2×:** the prediction is between half and double the true owners;
- **Correct owner range:** the prediction falls in the true SteamSpy range (13 ranges).

In [16]:
LEVELS = np.log1p(np.sort(train["owners"].unique()))            # the 13 SteamSpy range midpoints, log scale


def owner_range(log_values):
    return np.abs(np.asarray(log_values)[:, None] - LEVELS[None, :]).argmin(1)


def accuracy(frame, pred):
    y = frame["log_owners"].to_numpy()
    return {"Within 2x": np.mean(np.abs(pred - y) <= np.log(2)),
            "Correct owner range": np.mean(owner_range(pred) == owner_range(y)),
            "Within one range": np.mean(np.abs(owner_range(pred) - owner_range(y)) <= 1)}


constant = np.median(train["log_owners"])
print(f"Constant guess: {np.expm1(constant):,.0f} owners for every game")
check1 = {}
for label, frame, pred in [("One year ahead", ahead, ahead["pred"].to_numpy()), ("Validation", validation, validation["pred"].to_numpy())]:
    check1[(label, "Final model")] = accuracy(frame, pred)
    check1[(label, "Constant guess")] = accuracy(frame, np.full(len(frame), constant))
pd.DataFrame(check1).T.style.format("{:.1%}").set_caption("Check 1: how often a single forecast is close")

Constant guess: 10,000 owners for every game


**The final model is not more accurate than the constant guess for a typical game.** It lands in the correct owner range *less* often than simply predicting 10,000 owners for everything, because most games really do stay under 20,000 owners. It is only better at being "roughly right" (within one range).

**Check 2: can it pick out the games that will sell well?** Here the question is not the exact number but the ranking: are the games it rates highest the ones that reach 100,000 owners?

In [17]:
check2 = {}
for label, frame in [("One year ahead", ahead), ("Validation", validation)]:
    big = (frame["owners"] >= 100_000).to_numpy()
    top = (frame["pred"] >= frame["pred"].quantile(0.9)).to_numpy()
    check2[label] = {"Games with 100,000+ owners": big.mean(), "Among the model's top 10%": big[top].mean(),
                     "ROC-AUC for 100,000+": roc_auc_score(big, frame["pred"])}
pd.DataFrame(check2).T.style.format({"Games with 100,000+ owners": "{:.1%}", "Among the model's top 10%": "{:.1%}",
                                     "ROC-AUC for 100,000+": "{:.3f}"}).set_caption("Check 2: ranking the games")

,"Games with 100,000+ owners",Among the model's top 10%,"ROC-AUC for 100,000+"
One year ahead,5.5%,33.5%,0.865
Validation,10.7%,61.3%,0.893


**Ranking is where the model is useful.** Among the 10% of games it rates highest, the share that reaches 100,000 owners is about six times the overall rate. So the model can point to *which* games are likely to do well, even though it cannot say *how many* owners each will get.

**Check 3: how far off is it for the games that did sell well?** For every game that reached 100,000 owners, the prediction is compared with the truth, by the kind of history the game has.

In [18]:
check3 = {}
for label, frame in [("One year ahead", ahead), ("Validation", validation)]:
    big = frame[frame["owners"] >= 100_000]
    for kind in KINDS:
        part = big[big["analogy_type"] == kind]
        ratio = np.expm1(part["pred"]) / part["owners"]
        check3[(label, kind)] = {"Games": len(part), "Within 2x": np.mean(np.abs(part["pred"] - part["log_owners"]) <= np.log(2)),
                                 "Median prediction / truth": ratio.median()}
pd.DataFrame(check3).T.style.format({"Games": "{:,.0f}", "Within 2x": "{:.1%}", "Median prediction / truth": "{:.2f}"}).set_caption(
    "Check 3: games that reached 100,000+ owners")

In [19]:
bins = pd.cut(ahead["owners"], [0, 20_000, 100_000, 1_000_000, np.inf], right=False,
              labels=["Under 20,000", "20,000-100,000", "100,000-1 million", "Over 1 million"])
medians = ahead.assign(true_range=bins, history=np.where(ahead["analogy_type"] == "No predecessor", "No earlier game", "Has an earlier game"),
                       predicted=np.expm1(ahead["pred"]))
medians.pivot_table(index="true_range", columns="history", values="predicted", aggfunc="median", observed=True).style.format(
    "{:,.0f}").set_caption("One year ahead: median predicted owners by true owner range")

history,Has an earlier game,No earlier game
true_range,,
"Under 20,000","12,718","13,474"
"20,000-100,000","18,823","14,754"
"100,000-1 million","70,223","18,759"
Over 1 million,"432,590","69,555"


**This is the clearest result of the project.** For successful games with a series predecessor, the forecast is within 2× of the truth almost half the time (44–48%), and the typical prediction is 60–78% of the true owners. For successful games with no history, it is within 2× less than one time in six (7–15%), and the typical prediction is only 9–15% of the true owners. The median table shows the same thing: for games that sold over a million copies, the median forecast is about 430,000 with an earlier game but about 70,000 without one. In other words, **the model cannot see a new studio's hit coming**. Nothing on a store page signals it.

**What does the model rely on?** Permutation importance measures how much test-set R² drops when one variable is shuffled. The sums by group show which kind of information the final model uses.

In [20]:
final_model = HistGradientBoostingRegressor(random_state=RANDOM_STATE).fit(train[ANALOGY], train["log_owners"])
imp = permutation_importance(final_model, test[ANALOGY], test["log_owners"], n_repeats=5, random_state=RANDOM_STATE)
importance = pd.Series(imp.importances_mean, index=ANALOGY)
group_names = {"BASE": "Store page (42)", "STUDIO_AND_LEAD_TIME": "Studio and lead time (7)",
               "PREDECESSOR": "Predecessor owners and gaps (6)", "STUDIO_AVERAGES": "Studio's earlier sales (4)"}
display(pd.Series({group_names[g]: importance[v].sum() for g, v in groups.items()}, name="Drop in test R² (sum)").to_frame()
        .style.format("{:.3f}").set_caption("Importance by group"))
importance.sort_values(ascending=False).head(8).rename("Drop in test R²").to_frame().style.format("{:.3f}").set_caption(
    "The eight most important variables")

,Drop in test R² (sum)
Store page (42),0.386
Studio and lead time (7),0.029
Predecessor owners and gaps (6),0.014
Studio's earlier sales (4),0.107


,Drop in test R²
release_year,0.202
pub_prior_mean_log_owners,0.091
n_languages,0.061
genre_Free_To_Play,0.038
price,0.027
log_lead_days,0.025
dev_prior_mean_log_owners,0.013
cat_Steam_Cloud,0.008


**Reading the result.**

- **Release year is the single most important variable.** Part of this is real (the market is more crowded each year), but part comes from the data: owner counts are a 2025 snapshot, so older games have had more years to collect owners.
- **The studio's earlier sales come next**, especially the publisher's average. A game's history matters more than any single store-page detail except release year.
- The 42 store-page variables together still carry the most weight, but each one adds only a little.

## Step 10. Conclusions

**Answers to the research questions.**

| | Result |
|---|---|
| **RQ1:** How well can owners be predicted from release-day information? | About half of the variation in log owners for unseen games (R² 0.495 test, about 0.51 validation), and about a third one year ahead (0.312). The parent paper's method scores below zero. |
| **RQ2:** Does game history add more than store-page details? | Yes, history gives the largest gain from new information (one year ahead 0.257 → 0.312), but only for games that have history. |
| **H1** | Supported: much better than a constant guess at *ranking* games. Not supported for *exact* numbers: a single forecast is not more accurate than predicting 10,000 owners. |
| **H2** | Supported, with a limit: history helps a lot when a series predecessor exists (about 0.59 one year ahead) and not at all for the 52% of games with no earlier title. |

**Honest summary.** With the variables available on release day, I could not build an accurate forecast of how many owners a new Steam game will get. The only meaningful forecasts are for games with something to learn from, such as a predecessor in the same series or a studio with a sales record. For first games and for free-to-play games the model is close to guessing, and it badly underestimates surprise hits.

**Where I followed and where I left the parent papers.**

- **Parent paper 1:** kept its models and metrics, but removed its two leaks (regional sales and the sorted row index) and added time-based testing.
- **Parent paper 2:** kept the log target and percentage reporting; separate paid/free models helped Linear Regression but not Gradient Boosting; its wishlist data is not public, so its main variable could not be used.
- **Parent paper 3:** kept the predecessor idea and extended it from franchises to developers and publishers across all of Steam; its Google Trends signal was tested in a 30-game pilot and was missing for almost all small and mid-sized games.

**Limitations.** Owners are SteamSpy range estimates, not real sales. They are a 2025 snapshot, so older games and predecessors had more time to sell. No pre-release interest measure (wishlists, followers, search volume) exists for most games.

**Future work.** Add the text of the store description, add features from the cover image, and report a prediction range instead of a single number, so users can see how uncertain each forecast is.